# Barrido automático de punta a punta

`keysight_SA_control.ipynb` mide un `barrido X` a la vez: para cada uno hay que
reconfigurar a mano el span/RBW del analizador y volver a correr la celda del
loop. Así se terminó midiendo en 9 sesiones separadas a lo largo de varias
horas (una con casi 2 h de hueco en el medio).

En `analisis_barridos_frecuencia.ipynb` vimos que eso deja una marca en los
datos: cada sesión queda a un nivel de amplitud levemente distinto de la
siguiente (saltos de 0.04-0.12 dB), probablemente por el equipo
calentándose/estabilizándose de forma distinta en cada sesión.

Este notebook hace **todo el barrido en una sola corrida**: recorre una lista
de frecuencias de prueba y, para cada una, calcula automáticamente el
`f_start`/`f_stop`/`RBW` del analizador en vez de tenerlos fijos por carpeta.
Así no hace falta parar a reconfigurar nada a mano entre sub-rangos.

**Importante:** esto controla el equipo real. No se puede probar sin estar
conectado en el labo. Antes de lanzar el barrido completo (puede tardar bastante),
correlo primero con una lista corta de 5-10 frecuencias (sección de abajo) para
chequear que el span/RBW elegidos automáticamente anden bien.

In [ ]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import class_spectrum_analyzer as cspean
import class_signal_generator as csiggen

## Conexión

Mismas IPs que `keysight_SA_control.ipynb`. `base_path` es relativo (la
carpeta de arriba de `programar/`), para que ande igual en cualquier máquina
donde se clone el repo.

In [ ]:
sa_connection_info = 'TCPIP0::172.31.0.31::inst0::INSTR'
gen_connection_info = 'TCPIP0::172.31.0.20::inst0::INSTR'
base_path = Path.cwd().parent

keysight_N9021B = cspean.spectrum_analyzer(sa_connection_info,
                                           force_stay_open=True,
                                           storage_path=str(base_path) + "/")
keysight_N9021B.set_mode("SA")
keysight_N9021B.set_timeout(60000)

picotest_G5100A = csiggen.signal_generator(gen_connection_info)

## Parámetros del barrido

`nombre_carpeta` define dónde se guardan los `.csv` -- tiene que empezar con
`barrido` para que `analisis_barridos_frecuencia.ipynb` la encuentre sola
después. Usá un nombre distinto por campaña (por ejemplo
`barrido_con_filtro_ambiente`) para no pisar corridas anteriores.

La lista de frecuencias sale de una grilla logarítmica (`puntos_por_decada`
puntos por década), en vez de tramos lineales por sub-rango como antes -- da
una densidad de puntos pareja en todo el barrido.

In [ ]:
nombre_carpeta = "barrido_automatico"
sweep_amplitude_vpp = 0.2

f_test_min_hz = 100.0
f_test_max_hz = 14.8e6
puntos_por_decada = 40

n_decadas = np.log10(f_test_max_hz / f_test_min_hz)
n_puntos = int(round(n_decadas * puntos_por_decada)) + 1
frecuencias_test = np.geomspace(f_test_min_hz, f_test_max_hz, n_puntos)

print(f"{len(frecuencias_test)} frecuencias de prueba, de {f_test_min_hz/1e3:.3g} kHz "
      f"a {f_test_max_hz/1e6:.3g} MHz")

## Configuración del analizador para cada frecuencia

En vez de un span fijo por carpeta (lo que en el análisis anterior dejaba
resoluciones muy dispares entre carpetas, y `f_start=0` en la primera --
sospechosa de causar el offset raro que se vio ahí), el span queda centrado
en `f_test` y proporcional a `f_test` (`span_factor`). Así la resolución
relativa es parecida en todo el barrido y nunca toca 0 Hz.

`rbw = span / rbw_relativo` deja el RBW en un puñado de bins de ancho.
**`vbw = rbw`** (acoplado 1:1) es una prueba para ver si evita el patrón de
bins alternados que se encontró en 7 de las 9 carpetas viejas -- no está
confirmado que sea la causa, pero es lo primero para probar.

In [ ]:
def elegir_configuracion_sa(f_test_hz, span_factor=0.2, span_minimo_hz=50.0,
                             rbw_relativo=200, n_points=1001):
    span = max(f_test_hz * span_factor, span_minimo_hz)
    f_start = max(f_test_hz - span / 2, 1.0)
    f_stop = f_start + span
    rbw = span / rbw_relativo
    vbw = rbw
    return {"f_start": f_start, "f_stop": f_stop, "rbw": rbw, "vbw": vbw, "n_points": n_points}


def formatear_khz(f_test_hz):
    # cantidad de decimales que ajusta con la magnitud, para no perder
    # precision en frecuencias chicas ni generar decimales de mas en las
    # grandes -- con puntos_por_decada=40 dos frecuencias consecutivas
    # difieren ~6%, así que con esto nunca da el mismo nombre de archivo
    # para dos frecuencias distintas.
    khz = f_test_hz / 1e3
    decimales = max(0, 3 - int(np.floor(np.log10(max(khz, 1e-9)))))
    return f"{khz:.{decimales}f}"


# vista previa para un par de frecuencias, antes de tocar el equipo
for f in (100.0, 10e3, 1e6, 14.8e6):
    print(f"f_test={f/1e3:>10.3g} kHz -> {elegir_configuracion_sa(f)} -> sweep_f_test_{formatear_khz(f)}kHz.csv")

## Prueba chica antes del barrido completo

Mide de verdad, pero solo 8 frecuencias representativas de todo el rango, en
su propia carpeta (`barrido_automatico_test`) para no mezclarse con el
barrido completo. Al terminar, `revisar_picos` grafica el pico de cada una --
tiene que verse una curva razonable (sin saltos enormes), y en cada `.csv`
individual el pico angosto y centrado, sin el patrón de diente de sierra en
el piso de ruido. Si algo no anda bien, ajustá `span_factor`/`rbw_relativo`
de `elegir_configuracion_sa` (celda de arriba) y repetí esta celda.

In [ ]:
frecuencias_prueba = np.geomspace(f_test_min_hz, f_test_max_hz, 8)

carpeta_prueba = correr_barrido(frecuencias_prueba, "barrido_automatico_test")
resultados_prueba = revisar_picos(carpeta_prueba)
resultados_prueba

## Barrido principal

Recién correr esto después de validar la prueba chica de arriba. Por cada
frecuencia: configura el SA (span centrado en `f_test`), prende el generador,
adquiere, guarda el `.csv`, apaga el generador y sigue con la siguiente --
todo en la misma corrida, sin pausas manuales. Al terminar, `revisar_picos`
grafica los picos de todo el barrido completo para chequear que da una curva
continua.

Si una frecuencia falla (por ejemplo, un timeout de VISA puntual), se anota
el error y se sigue con las demás en vez de cortar todo el barrido.

In [ ]:
carpeta_salida = correr_barrido(frecuencias_test, nombre_carpeta)
resultados_completos = revisar_picos(carpeta_salida)
resultados_completos

## Barrido principal

Por cada frecuencia: configura el SA (span centrado en `f_test`), prende el
generador a esa frecuencia, adquiere, guarda el `.csv`, apaga el generador y
sigue con la siguiente -- todo en la misma corrida, sin pausas manuales.

Si una frecuencia falla (por ejemplo, un timeout de VISA puntual), se anota
el error y se sigue con las demás en vez de cortar todo el barrido.

In [ ]:
carpeta_salida = base_path / nombre_carpeta
carpeta_salida.mkdir(exist_ok=True)
ya_guardados = list(carpeta_salida.glob("sweep_f_test_*.csv"))
if ya_guardados:
    print(f"Aviso: {carpeta_salida} ya tiene {len(ya_guardados)} archivos -- "
          "los que se repitan en esta corrida se van a pisar.")

errores = []
t0 = time.time()

for i, f_test in enumerate(frecuencias_test):
    config = elegir_configuracion_sa(f_test)
    keysight_N9021B.set_device_configuration(**config)
    time.sleep(0.2)  # margen para que el SA termine de retunear (LO, filtros)

    picotest_G5100A.set_waveform("sine", frequency=f_test, amplitude=sweep_amplitude_vpp, offset=0)
    picotest_G5100A.output_on()
    time.sleep(0.5)  # margen para que la señal se estabilice

    try:
        keysight_N9021B.acquire()
        nombre_archivo = f"{nombre_carpeta}/sweep_f_test_{formatear_khz(f_test)}kHz.csv"
        keysight_N9021B.save_data(nombre_archivo)
    except Exception as exc:
        print(f"  ERROR en f_test={f_test/1e3:.4g} kHz: {exc}")
        errores.append((f_test, str(exc)))
    finally:
        picotest_G5100A.output_off()

    transcurrido = time.time() - t0
    restante_estimado = transcurrido / (i + 1) * (len(frecuencias_test) - i - 1)
    print(f"[{i+1}/{len(frecuencias_test)}] f_test={f_test/1e3:.4g} kHz -- "
          f"{transcurrido/60:.1f} min transcurridos, "
          f"~{restante_estimado/60:.1f} min restantes")

print(f"\nListo. {len(frecuencias_test) - len(errores)}/{len(frecuencias_test)} puntos guardados en {carpeta_salida}")
if errores:
    print(f"{len(errores)} fallaron:")
    for f, err in errores:
        print(f"  {f/1e3:.4g} kHz: {err}")

## Después de medir

`revisar_picos` de arriba es solo un vistazo rápido. Para el análisis
completo (chequeo de offset en frecuencia, patrón de bins alternados, ripple,
etc.) correr `analisis_barridos_frecuencia.ipynb` sin tocar nada: encuentra
sola cualquier carpeta `barrido*` nueva, `barrido_automatico` incluida.

In [ ]:
keysight_N9021B.close_device()
picotest_G5100A.close_device()

## Después de medir

Correr `analisis_barridos_frecuencia.ipynb` sin tocar nada: encuentra solo
cualquier carpeta `barrido*` nueva, `barrido_automatico` incluida.